### Chapter 2.1 - Data Manipulation

A machine learning model does not directly understand images, sentences, tables, or sounds. It receives numbers arranged in structured containers. In this chapter, the container we care about is the tensor.

A tensor is a grid of numbers with any number of dimensions. A single number is a 0-dimensional tensor. A list of numbers is a 1-dimensional tensor. A table of numbers is a 2-dimensional tensor. A stack of tables is a 3-dimensional tensor.

This chapter builds tensor manipulation in layers: first plain Python containers, then NumPy arrays, then PyTorch tensors.


### 2.1.1 Getting Started

#### 1. Intuition

Data manipulation means creating, inspecting, reshaping, and moving around numerical data before a model uses it.

Think of a tensor as a spreadsheet-like container. A vector is like one row of a spreadsheet. A matrix is like a full spreadsheet. A higher-dimensional tensor is like several spreadsheets stacked together.

Before a model can learn, we must answer simple questions about the data:

- How many numbers do we have?
- How are those numbers arranged?
- Can we reshape the arrangement without changing the numbers?
- Are the numbers stored in a format the ML framework can compute with?

#### 2. Why this exists

Machine learning code often fails because the numbers are arranged in the wrong shape, not because the math idea is wrong.

For example, a model may expect 4 input features per example, but your data may accidentally be shaped as 4 examples with 1 feature each. Both contain 4 numbers, but they mean different things.

A shape tells us the size of each dimension. A dimension is one direction you can move through the data. For a table, rows are one dimension and columns are another dimension.


#### 3. Examples

First, use plain Python. A Python list stores values in order. A nested list stores lists inside a list, which can represent a table.


In [ ]:
x = [0, 1, 2, 3]
len(x)


In [ ]:
table = [[0, 1, 2], [3, 4, 5]]
rows = len(table)
cols = len(table[0])
(rows, cols)


NumPy gives us arrays. An array is a numerical container designed for fast computation. It stores shape information directly.


In [ ]:
import numpy as np

x_np = np.arange(6)
x_np.shape


In [ ]:
X_np = x_np.reshape(2, 3)
X_np.shape


PyTorch gives us tensors. A PyTorch tensor is like a NumPy array, but it is designed for deep learning. It can run on CPUs or GPUs, and later it can track operations for automatic differentiation. Automatic differentiation means the framework can compute slopes needed for learning by remembering how values were produced.


In [ ]:
import torch

x = torch.arange(6)
X = x.reshape(2, 3)
X.shape


In [ ]:
zeros = torch.zeros((2, 3))
ones = torch.ones((2, 3))
random = torch.randn((2, 3))
zeros.shape, ones.shape, random.shape


#### 4. Step-by-step breakdown

`torch.arange(6)` creates a 1-dimensional tensor containing six numbers: 0 through 5.

`reshape(2, 3)` reinterprets the same six numbers as 2 rows and 3 columns. It does not invent new data. It only changes the arrangement.

`torch.zeros((2, 3))` creates a 2 by 3 tensor filled with 0. This is useful when you need a placeholder.

`torch.ones((2, 3))` creates a 2 by 3 tensor filled with 1. This is useful for masks, initial values, and simple checks.

`torch.randn((2, 3))` creates a 2 by 3 tensor with random values sampled from a bell-shaped distribution centered near 0. Random tensors are often used to initialize model parameters before learning begins.

#### 5. Connection to ML systems

A batch of training data is usually a tensor. A batch is a small group of examples processed together. For example, if we have 4 examples and each example has 3 features, the data tensor shape is usually `(4, 3)`.

Shape errors are among the most common early ML bugs. The framework does not know what you intended. It only sees dimensions and numbers.

#### 6. Common confusion points

- `reshape` changes arrangement, not the values.
- `(2, 3)` means 2 rows and 3 columns for a matrix-like tensor.
- `shape` is metadata. Metadata is information about the data, not the data values themselves.
- A tensor is not automatically a neural network object. It is just the numerical container neural networks use.


### 2.1.2 Indexing and Slicing

#### 1. Intuition

Indexing means selecting one item from a container by position. Slicing means selecting a range of items.

Python counts positions starting from 0. The first item is at index 0, the second item is at index 1, and so on.

For a 2-dimensional tensor, we usually use two positions: one for the row and one for the column.

#### 2. Why this exists

Machine learning workflows constantly select parts of data:

- one example from a dataset
- one feature column from a table
- one image from a batch
- one range of time steps from a sequence

Without indexing and slicing, every operation would need to process the whole dataset even when we only need a small part.


#### 3. Examples

Plain Python list indexing uses square brackets. A slice uses `start:stop`, where `stop` is excluded.


In [ ]:
values = [10, 20, 30, 40]
first = values[0]
middle = values[1:3]
first, middle


In [ ]:
table = [[1, 2, 3], [4, 5, 6]]
first_row = table[0]
one_value = table[1][2]
first_row, one_value


NumPy and PyTorch let us index multiple dimensions in one bracket expression.


In [ ]:
X_np = np.array([[1, 2, 3], [4, 5, 6]])
row = X_np[1]
col = X_np[:, 2]
row, col


In [ ]:
X = torch.tensor([[1, 2, 3], [4, 5, 6]])
top_left = X[0, 0]
last_column = X[:, 2]
top_left, last_column


We can also write into selected positions. This is called assignment. Assignment means replacing the value stored at a location.


In [ ]:
X = torch.zeros((2, 3))
X[0, 1] = 9
X[:, 2] = 5
X


#### 4. Step-by-step breakdown

`values[0]` selects the first item because Python starts counting at 0.

`values[1:3]` selects positions 1 and 2. It stops before position 3.

`X[:, 2]` selects every row from column 2. The colon `:` means all positions along that dimension.

`X[0, 1] = 9` changes the value in row 0, column 1.

`X[:, 2] = 5` changes the value in column 2 for every row.

#### 5. Connection to ML systems

Indexing appears when selecting examples, labels, features, and model predictions. If `X` stores input examples and `y` stores labels, then `X[0]` and `y[0]` usually belong to the same training example.

#### 6. Common confusion points

- Slices exclude the stop index.
- `:` means everything along that dimension.
- In a 2-dimensional tensor, `X[0]` usually means the first row, not the first column.
- Assignment changes stored values. It is not just viewing values.


### 2.1.3 Operations

#### 1. Intuition

An operation takes one or more values and produces a result. Addition, multiplication, exponentiation, and comparison are operations.

An elementwise operation applies the same operation separately to each matching position.

If `x = [1, 2, 3]`, then adding 10 elementwise gives `[11, 12, 13]`.

#### 2. Why this exists

ML models apply many repeated numerical operations. Doing them one value at a time in Python is slow and noisy to read. Tensor libraries express the whole operation at once.

This is useful for both correctness and speed. The code says what mathematical operation should happen, and the library decides how to execute it efficiently.


#### 3. Examples

Plain Python lists do not automatically do elementwise math. We write the loop ourselves.


In [ ]:
x = [1, 2, 3]
y = [10, 20, 30]
result = []
for i in range(len(x)):
    result.append(x[i] + y[i])
result


A list comprehension is a compact Python way to build a list from a loop. It is still plain Python, not a tensor operation.


In [ ]:
x = [1, 2, 3]
squared = [value * value for value in x]
squared


NumPy and PyTorch apply operations elementwise when shapes match.


In [ ]:
x_np = np.array([1, 2, 3])
y_np = np.array([10, 20, 30])
x_np + y_np


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])
x + y, x * y, torch.exp(x)


Concatenation joins tensors along a dimension. To concatenate means to attach containers end to end.


In [ ]:
A = torch.arange(6).reshape(2, 3)
B = torch.ones((2, 3))
rows_joined = torch.cat((A, B), dim=0)
cols_joined = torch.cat((A, B), dim=1)
rows_joined.shape, cols_joined.shape


#### 4. Step-by-step breakdown

In the manual loop, `range(len(x))` creates positions 0, 1, and 2. Each loop step reads one value from `x`, one value from `y`, adds them, and appends the answer to `result`.

`x_np + y_np` asks NumPy to do the same matching-position addition without writing the loop yourself.

`x + y` in PyTorch performs elementwise addition.

`x * y` performs elementwise multiplication, not a dot product and not matrix multiplication.

`torch.exp(x)` applies the exponential function to every element. The exponential function maps a number `a` to `e` raised to `a`, where `e` is approximately 2.718.

`torch.cat((A, B), dim=0)` stacks tensors by adding more rows. `dim=0` means the first dimension.

`torch.cat((A, B), dim=1)` stacks tensors by adding more columns. `dim=1` means the second dimension.

#### 5. Connection to ML systems

Neural networks are built from tensor operations. A layer takes input tensors, combines them with parameter tensors, applies operations, and returns output tensors.

Elementwise operations appear in activation functions, loss functions, masks, normalization, and data preprocessing.

#### 6. Common confusion points

- `*` on PyTorch tensors is elementwise multiplication.
- Elementwise operations require compatible shapes.
- `dim=0` and `dim=1` are dimension numbers, not values inside the tensor.
- Concatenation changes shape because it joins containers together.


### 2.1.4 Broadcasting

#### 1. Intuition

Broadcasting is a rule that lets tensor libraries combine arrays with different but compatible shapes.

The intuition is copying without actually copying. If one tensor is missing a dimension or has size 1 along a dimension, the library can act as if the smaller tensor were repeated to match the larger one.

#### 2. Why this exists

Broadcasting avoids manual loops and avoids physically copying data many times. It lets us write the math in a compact way.

For example, adding the same bias value to every row of a table is common in neural networks. Broadcasting lets us write that as one tensor addition.


#### 3. Examples

Plain Python version: manually add a row vector to every row of a table.


In [ ]:
table = [[1, 2, 3], [4, 5, 6]]
bias = [10, 20, 30]
out = []
for row in table:
    out.append([row[i] + bias[i] for i in range(3)])
out


NumPy and PyTorch do this automatically when the shapes are compatible.


In [ ]:
table_np = np.array([[1, 2, 3], [4, 5, 6]])
bias_np = np.array([10, 20, 30])
table_np + bias_np


In [ ]:
A = torch.arange(3).reshape(3, 1)
B = torch.arange(2).reshape(1, 2)
A + B


#### 4. Step-by-step breakdown

The manual Python loop visits each row. For each row, the list comprehension adds the matching bias value to each column.

In `table_np + bias_np`, the table has shape `(2, 3)` and the bias has shape `(3,)`. NumPy treats the bias as if it were repeated once for each row.

In the PyTorch example, `A` has shape `(3, 1)` and `B` has shape `(1, 2)`.

PyTorch acts as if `A` were repeated across columns and `B` were repeated across rows. The result has shape `(3, 2)`.

The two basic broadcasting checks are:

- dimensions match exactly, or
- one of the dimensions has size 1 and can be repeated logically.

#### 5. Connection to ML systems

Broadcasting appears when adding bias terms, applying normalization statistics, masking outputs, and comparing predictions to labels.

A bias is a learnable number added to a model's computed score. If a layer produces many rows of outputs, broadcasting lets one bias vector be added to every row.

#### 6. Common confusion points

- Broadcasting does not always copy data into memory. It often behaves like virtual repetition.
- Shape `(3,)` and shape `(3, 1)` are different.
- Broadcasting follows rules; it is not guessing your intention.
- If dimensions are incompatible, the operation fails instead of silently inventing a meaning.


### 2.1.5 Saving Memory

#### 1. Intuition

A tensor occupies memory. Memory is the computer storage used while a program is running.

Some operations create a new tensor to store the result. Other operations update an existing tensor in place. In place means the old storage is reused instead of allocating new storage.

#### 2. Why this exists

Deep learning can use very large tensors. Creating unnecessary copies can waste memory and slow the program.

However, in-place modification can also make code harder to reason about because old values disappear. Use it when the memory benefit is clear and the code remains understandable.


#### 3. Examples

Plain Python names refer to objects. Two names can point to the same object. The `id` function returns an identity number for an object during the current program run.


In [ ]:
a = [1, 2, 3]
b = a
b[0] = 99
a


PyTorch lets us compare whether a tensor name still points to the same object after an operation.


In [ ]:
X = torch.ones((2, 3))
Y = torch.zeros((2, 3))
before = id(Y)
Y = X + Y
before == id(Y)


In [ ]:
X = torch.ones((2, 3))
Y = torch.zeros((2, 3))
before = id(Y)
Y[:] = X + Y
before == id(Y)


PyTorch also has operation names ending in an underscore. By convention, many PyTorch methods ending in `_` modify the tensor in place.


In [ ]:
Y = torch.zeros((2, 3))
before = id(Y)
Y.add_(1)
before == id(Y)


#### 4. Step-by-step breakdown

`b = a` does not copy the list. It gives the same list a second name.

`b[0] = 99` changes the shared list, so `a` shows the change too.

`Y = X + Y` creates a new tensor result, then makes the name `Y` point to that new object.

`Y[:] = X + Y` writes the result into the existing storage owned by `Y`.

`Y.add_(1)` adds 1 directly into `Y`. The underscore warns us that the method modifies the object in place.

#### 5. Connection to ML systems

Memory matters when training large models because activations, parameters, gradients, and optimizer state may all need to be stored.

A gradient is a collection of slopes telling the model how changing each parameter would change the loss. A loss is a number measuring how wrong the model is. Optimizer state is extra information an update rule stores to decide how to change parameters.

In-place operations can reduce memory use, but they must be used carefully when automatic differentiation is tracking values needed to compute gradients.

#### 6. Common confusion points

- Reassigning a name is not the same as modifying an object.
- `Y = X + Y` usually allocates a new tensor.
- `Y[:] = ...` modifies the existing tensor contents.
- PyTorch methods ending in `_` usually mean in-place modification.


### 2.1.6 Conversion to Other Python Objects

#### 1. Intuition

Conversion means changing data from one container type to another while preserving the underlying values as much as possible.

Common containers are Python lists, NumPy arrays, and PyTorch tensors.

#### 2. Why this exists

Different libraries expect different input types. A plotting library might prefer NumPy arrays. A deep learning model expects PyTorch tensors. Plain Python code often uses lists.

Conversion lets these tools cooperate.


#### 3. Examples

Start with a Python list, convert it to NumPy, then convert it to PyTorch.


In [ ]:
values = [1, 2, 3]
array = np.array(values)
tensor = torch.tensor(values)
type(values), type(array), type(tensor)


Convert between NumPy and PyTorch.


In [ ]:
array = np.array([1.0, 2.0, 3.0])
tensor = torch.from_numpy(array)
back_to_array = tensor.numpy()
type(tensor), type(back_to_array)


A scalar tensor contains one value. The `.item()` method extracts that one value as a plain Python number.


In [ ]:
score = torch.tensor(3.5)
python_number = score.item()
type(python_number), python_number


#### 4. Step-by-step breakdown

`np.array(values)` reads the Python list and creates a NumPy array.

`torch.tensor(values)` reads the Python list and creates a new PyTorch tensor.

`torch.from_numpy(array)` creates a tensor from a NumPy array. This may share memory with the NumPy array when possible, so changing one can affect the other.

`tensor.numpy()` converts a CPU tensor to a NumPy array.

`score.item()` extracts a plain Python number from a tensor that contains exactly one value.

#### 5. Connection to ML systems

Training code usually keeps data as tensors. Visualization, logging, and debugging often convert tensors to Python or NumPy objects.

For example, a training loop may compute a tensor loss, then call `.item()` so the loss can be printed as a normal number.

#### 6. Common confusion points

- `.item()` only works cleanly when the tensor has one value.
- NumPy arrays do not track PyTorch gradients.
- GPU tensors must usually be moved to CPU before converting to NumPy.
- `torch.tensor(existing_data)` usually copies data; `torch.from_numpy(array)` may share memory.


### 2.1.7 Summary

#### 1. Intuition

Data manipulation is the skill of putting numbers into the right container, shape, and location before computation.

#### 2. Why this exists

Machine learning models are strict about numerical structure. The same values can mean different things depending on shape and dimension order.

#### 3. Examples

- Create tensors with `torch.arange`, `torch.zeros`, `torch.ones`, and `torch.randn`.
- Inspect shape with `.shape`.
- Rearrange values with `.reshape`.
- Select values with indexing and slicing.
- Combine values with elementwise operations.
- Rely on broadcasting only when shapes are compatible.
- Save memory with in-place operations when appropriate.
- Convert between Python lists, NumPy arrays, and PyTorch tensors.

#### 4. Step-by-step breakdown

The workflow is usually: create or load data, inspect its shape, select the needed part, apply operations, and convert only when another library requires a different type.

#### 5. Connection to ML systems

Every later model will use these ideas. Inputs, labels, predictions, losses, parameters, and gradients are all represented as tensors or values derived from tensors.

#### 6. Common confusion points

- A tensor's shape is part of its meaning.
- A tensor operation may create a new object unless it is explicitly in place.
- Broadcasting is useful but must be checked carefully.
- Framework code becomes easier to read when tensor shape changes are tracked step by step.


### 2.1.8 Exercises

#### 1. Intuition

Exercises are small tests of whether you can manipulate data deliberately instead of copying code mechanically.

#### 2. Why this exists

Tensor manipulation becomes natural only through repeated shape-aware practice. The goal is not to memorize commands. The goal is to predict what each command does to values, shape, and memory.


#### 3. Examples

Exercise 1: Create a tensor with values 0 through 11, reshape it into 3 rows and 4 columns, then inspect its shape.


In [ ]:
X = torch.arange(12)
X = X.reshape(3, 4)
X.shape


Exercise 2: Select the second row and the last column from the tensor.


In [ ]:
second_row = X[1]
last_column = X[:, 3]
second_row, last_column


Exercise 3: Add a bias vector to every row using broadcasting.


In [ ]:
bias = torch.tensor([100, 200, 300, 400])
Y = X + bias
Y


Exercise 4: Convert a tensor with one value into a plain Python number.


In [ ]:
loss = torch.tensor(2.75)
loss_value = loss.item()
type(loss_value), loss_value


#### 4. Step-by-step breakdown

Exercise 1 checks whether you understand that 12 values can be arranged as 3 by 4 because 3 times 4 equals 12.

Exercise 2 checks row selection and column selection. `X[1]` selects the second row. `X[:, 3]` selects all rows from the fourth column.

Exercise 3 checks broadcasting. `X` has shape `(3, 4)` and `bias` has shape `(4,)`, so the bias is applied to every row.

Exercise 4 checks scalar extraction. `.item()` turns a one-value tensor into a plain Python number.

#### 5. Connection to ML systems

These exercises resemble real ML setup work: reshape data, select features, add bias-like values, and log scalar losses.

#### 6. Common confusion points

- A 3 by 4 tensor has 12 values.
- Row 1 is the second row because indexing starts at 0.
- The last column in a 4-column tensor has index 3.
- A tensor can display like a number while still being a tensor object.
